# Run Broadcasting Experiments

Use this notebook for the main protocol workflow: exact simulation, QEC Monte Carlo sampling, a single IBM hardware point, or an IBM hardware tau sweep. Results are saved through the unified JSON schema in `results/`.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from broadcasting import (
    ExactBackend,
    HardwareBackend,
    HPCBackend,
    ProtocolConfig,
    SamplingBackend,
    load_run,
    save_run,
)
from broadcasting.plotting import (
    periodogram_from_run,
    plot_fidelity_vs_noise,
    plot_periodicity_comparison,
    plot_run_sweep,
    save_figure,
)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120})


## Configuration

Set `MODE` to `exact`, `sampling`, `hardware`, `hardware_tau_sweep`, or `hpc`. Every mode builds the
same `ProtocolConfig` and hands it to whichever `Backend` subclass matches (`ExactBackend`,
`SamplingBackend`, `HardwareBackend`, `HPCBackend`).


In [2]:
MODE = "exact"

M = 1
N = 2
alpha = 1.0 / np.sqrt(2)
use_qec = False
outcomes = [0] * M
seed = 42

rng = np.random.default_rng(seed)
nt = 1
theta_samples = rng.uniform(0, 2 * np.pi, size=(nt, M)).tolist()
thetas = theta_samples[0]

p_list = np.linspace(0, 1, 50).tolist()
n_samples = 1000

tau_values = np.linspace(0, 6000, 121).astype(int).tolist()
tau = tau_values[0]

IBM_PROFILE = "mprest1"
IBM_BACKEND = None  # None -> least-busy operational backend (Fig. 5 does not depend on which one)
OPTIMIZATION_LEVEL = 3
SHOTS = 4096

# Manual yes/no toggle for Runtime's built-in Sampler-level dynamical decoupling
# (SamplerOptions.dynamical_decoupling.enable); see HardwareBackend._sampler().
DYNAMICAL_DECOUPLING = False

# Only used when MODE == "hpc": which Backend the cluster job itself should run,
# and whether to actually call `sbatch` (False just builds/prints the command).
HPC_MODE = "exact"
HPC_SUBMIT = False
HPC_ARRAY = False
HPC_CONCURRENCY = None

SAVE_FIGURES = False

FIGURE_DIR = Path("figures")

print(f"Mode={MODE}  M={M}  N={N}  QEC={use_qec}")
print("theta samples:")
for i, sample in enumerate(theta_samples):
    print(f"  {i}: {np.array(sample)}")

Mode=exact  M=1  N=2  QEC=False
theta samples:
  0: [4.86290927]


## Run


In [3]:
if MODE == "sampling" and not use_qec:
    raise ValueError("MODE='sampling' requires use_qec=True.")

config = ProtocolConfig(
    M=M,
    N=N,
    alpha=alpha,
    thetas=thetas,
    p_list=p_list if MODE in {"exact", "sampling", "hpc"} else [],
    use_qec=use_qec,
    outcomes_list=outcomes,
    tau=tau if MODE == "hardware" else None,
    n_samples=n_samples if MODE in {"sampling", "hpc"} else None,
    seed=seed,
)

if MODE == "exact":
    backend = ExactBackend()
    result = backend.run(config)
elif MODE == "sampling":
    backend = SamplingBackend(n_samples=n_samples, seed=seed)
    result = backend.run(config)
elif MODE == "hpc":
    backend = HPCBackend(mode=HPC_MODE, array=HPC_ARRAY, concurrency=HPC_CONCURRENCY, submit=HPC_SUBMIT)
    result = backend.run(config)
    print(result.metadata["command"])
elif MODE in {"hardware", "hardware_tau_sweep"}:
    from qiskit_ibm_runtime import QiskitRuntimeService

    service = QiskitRuntimeService(name=IBM_PROFILE)
    backend = HardwareBackend(
        service=service,
        backend_name=IBM_BACKEND,
        shots=SHOTS,
        optimization_level=OPTIMIZATION_LEVEL,
        dynamical_decoupling=DYNAMICAL_DECOUPLING,
    )
    if MODE == "hardware":
        result = backend.run(config)
    else:
        result = backend.run_tau_sweep(config, tau_values, theta_samples=theta_samples)
else:
    raise ValueError(f"Unknown MODE: {MODE}")



print(f"Recorded mode: {result.metadata.get('mode')}")
print(f"Fidelity array shape: {np.asarray(result.fidelities).shape}")

Recorded mode: exact
Fidelity array shape: (50, 2)


## Save And Plot


In [ ]:
out_path = save_run(result, config)
saved_run = load_run(out_path)
print(f"Saved to {out_path}")

if saved_run["sweep"]["axis"] == "p":
    fids = np.asarray(saved_run["fidelities"], dtype=float)
    fig = plot_fidelity_vs_noise(
        np.asarray(saved_run["sweep"]["values"], dtype=float),
        {f"Receiver {i}": fids[:, i] for i in range(saved_run["N"])},
        mode_label=saved_run.get("backend", MODE),
        protocol_info={"M": M, "N": N, "use_qec": use_qec},
        show=False,
    )
else:
    fig = plot_run_sweep(
        saved_run,
        tau_scale=4e-3,
        tau_label="Idle delay (us)",
        show=False,
    )

plt.tight_layout()
if SAVE_FIGURES:
    figure_path = FIGURE_DIR / f"{Path(out_path).stem}.png"
    save_figure(fig, figure_path)
    print(f"Saved title-free figure to {figure_path}")
plt.show()


## Optional Exact Vs Sampling Overlay

`SamplingBackend` is for the QEC path, so this comparison runs only when `use_qec=True`.


In [ ]:
RUN_COMPARISON = False

if RUN_COMPARISON and use_qec:
    compare_config = ProtocolConfig(
        M=M,
        N=N,
        alpha=alpha,
        thetas=thetas,
        p_list=p_list,
        use_qec=True,
        outcomes_list=outcomes,
        seed=seed,
    )
    exact = ExactBackend().run(compare_config)
    sampled = SamplingBackend(n_samples=5000, seed=seed).run(compare_config)

    p_arr = np.asarray(p_list, dtype=float)
    exact_avg = np.asarray(exact.fidelities, dtype=float).mean(axis=1)
    sampled_avg = np.asarray(sampled.fidelities, dtype=float).mean(axis=1)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(p_arr, exact_avg, label="Exact")
    ax.plot(p_arr, sampled_avg, "--", label="Sampling")
    ax.axhline(0.5, color="gray", linestyle=":", alpha=0.5)
    ax.set_xlabel("Depolarizing probability p")
    ax.set_ylabel("Average fidelity")
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(alpha=0.25)
    plt.tight_layout()
    if SAVE_FIGURES:
        save_figure(fig, FIGURE_DIR / "exact_vs_sampling.png")
    plt.show()
elif RUN_COMPARISON:
    print("Set use_qec=True before running the sampling comparison.")


## Optional Sampling Convergence


In [ ]:
RUN_CONVERGENCE = False

if RUN_CONVERGENCE:
    conv_config = ProtocolConfig(
        M=1,
        N=2,
        alpha=1.0 / np.sqrt(2),
        thetas=[0.0],
        p_list=np.linspace(0, 1, 21).tolist(),
        use_qec=True,
        outcomes_list=[0],
        seed=0,
    )
    exact_fids = np.asarray(ExactBackend().run(conv_config).fidelities)
    p_arr = np.asarray(conv_config.p_list)
    n_sweep = [50, 100, 200, 500, 1000, 2000, 5000, 10000]
    seeds = [0, 1, 2, 3, 4]

    # Multiple seeds so the fit isn't driven by one noisy trajectory realization.
    errors = np.zeros((len(seeds), len(n_sweep)))
    for si, seed in enumerate(seeds):
        for ni, ns in enumerate(n_sweep):
            sampled = SamplingBackend(n_samples=ns, seed=seed).run(conv_config)
            diff = np.abs(np.asarray(sampled.fidelities) - exact_fids)
            errors[si, ni] = np.trapz(diff, p_arr, axis=0).sum()
        print(f"seed={seed}: " + ", ".join(f"n={ns}:{errors[si, ni]:.4f}" for ni, ns in enumerate(n_sweep)))

    mean_errors = errors.mean(axis=0)
    std_errors = errors.std(axis=0)

    # Fit log(error) = slope*log(n) + intercept by linear regression, instead
    # of overlaying an assumed 1/sqrt(n) reference anchored to a single point --
    # this is what actually establishes the exponent rather than assuming it.
    log_n = np.log(n_sweep)
    log_err = np.log(mean_errors)
    slope, intercept = np.polyfit(log_n, log_err, 1)
    residuals = log_err - (slope * log_n + intercept)
    dof = len(n_sweep) - 2
    slope_se = (
        np.sqrt(np.sum(residuals ** 2) / dof / np.sum((log_n - log_n.mean()) ** 2))
        if dof > 0 else float("nan")
    )
    print(f"Fitted exponent: {slope:.3f} +/- {slope_se:.3f} (expect -0.5 for 1/sqrt(n) scaling)")

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.errorbar(
        n_sweep, mean_errors, yerr=std_errors, fmt="o", capsize=3,
        label=f"Observed ({len(seeds)} seeds, mean +/- std)",
    )
    fit_line = np.exp(intercept) * np.asarray(n_sweep, dtype=float) ** slope
    ax.plot(n_sweep, fit_line, "--", color="gray", label=f"Fit: n^{slope:.3f} +/- {slope_se:.3f}")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Number of samples")
    ax.set_ylabel("Error area")
    ax.grid(True, which="both", alpha=0.25)
    ax.legend()
    plt.tight_layout()
    if SAVE_FIGURES:
        save_figure(fig, FIGURE_DIR / "sampling_convergence.png")
        save_figure(fig, Path("manuscript") / "mc_sampling_convergence.png")
    plt.show()



## Optional Figure 5 DD Periodicity Comparison

Recreates the Fig. 5 delay-vs-fidelity setup (the `M=1, N=2` protocol without QEC, swept over
`tau_values`) twice on the *same* hardware backend -- once with dynamical decoupling off and once
on -- then compares the fidelity-vs-delay traces with an autocorrelation and periodogram. This
checks whether the visually quasi-periodic "drop-out" feature described in the manuscript is
(a) actually periodic and (b) affected by DD, without asserting either conclusion in advance.

Uses `M`, `N`, `alpha`, `thetas`/`theta_samples`, `tau_values`, `outcomes`, `seed`, `IBM_PROFILE`,
`IBM_BACKEND`, `SHOTS`, `OPTIMIZATION_LEVEL` from the **Configuration** cell above -- set
`use_qec = False` there before running this. Gated behind `RUN_DD_COMPARISON` since it submits two
full hardware tau sweeps (i.e. roughly twice the cost of one **Run** cell in `hardware_tau_sweep`
mode).


In [ ]:
RUN_DD_COMPARISON = False

if RUN_DD_COMPARISON:
    if use_qec:
        raise ValueError("Figure 5 recreation requires use_qec=False in the Configuration cell.")

    from qiskit_ibm_runtime import QiskitRuntimeService

    dd_config = ProtocolConfig(
        M=M,
        N=N,
        alpha=alpha,
        thetas=thetas,
        p_list=[],
        use_qec=False,
        outcomes_list=outcomes,
        tau=None,
        seed=seed,
    )

    service = QiskitRuntimeService(name=IBM_PROFILE)
    # Resolve one concrete backend up front so both DD-off and DD-on sweeps run
    # on identical hardware -- otherwise least_busy() could pick different
    # backends between the two submissions and confound the comparison.
    chosen_backend = (
        service.backend(IBM_BACKEND)
        if IBM_BACKEND
        else service.least_busy(simulator=False, operational=True)
    )
    print(f"Using backend: {chosen_backend.name}")

    dd_runs = {}
    for label, dd_flag in [("dd_off", False), ("dd_on", True)]:
        backend = HardwareBackend(
            service=service,
            backend_name=chosen_backend.name,
            shots=SHOTS,
            optimization_level=OPTIMIZATION_LEVEL,
            dynamical_decoupling=dd_flag,
        )
        result = backend.run_tau_sweep(dd_config, tau_values, theta_samples=theta_samples)
        out_path = save_run(result, dd_config)
        print(f"Saved {label} run to {out_path}")
        dd_runs[label] = load_run(out_path)

    fig = plot_periodicity_comparison(
        [dd_runs["dd_off"], dd_runs["dd_on"]],
        labels=["DD off", "DD on"],
        tau_scale=4e-3,
        tau_label="Idle delay (us)",
        show=False,
    )
    plt.tight_layout()
    if SAVE_FIGURES:
        save_figure(fig, FIGURE_DIR / "dd_periodicity_comparison.png")
    plt.show()

    for label, run in dd_runs.items():
        freqs, power = periodogram_from_run(run)
        if len(freqs) > 1:
            peak_freq = freqs[1:][np.argmax(power[1:])]
            peak_period_us = (1.0 / peak_freq) * 4e-3 if peak_freq else float("inf")
            print(f"{label}: dominant periodogram peak at ~{peak_period_us:.2f} us period")

management.get:WARNING:2026-09-08 14:47:15,634: Loading saved account: mprest1


Using backend: ibm_marrakesh
